#**1. Zero-Shot Object Detection**
Zero-Shot Object Detection은 학습 데이터에 존재하지 않았던 객체를 텍스트 설명만으로 탐지하는 기술입니다. 기존 객체 탐지(Object Detection)는 학습할 때 정의한 고정된 클래스만 예측할 수 있습니다. 반면 Zero-Shot Detection은 텍스트 프롬프트를 이용해 새로운 클래스를 자유롭게 추가할 수 있습니다.

###**1. 기존 Object Detection의 한계**
일반적인 객체 탐지 모델(YOLO, Faster R-CNN 등)은 아래와 같은 구조를 가집니다.
~~~
이미지 → 특징 추출 → 바운딩 박스 예측 + 고정 클래스 분류
~~~

- 학습한 클래스만 탐지 가능
- 새로운 객체를 추가하려면 다시 학습해야 함
- 현실 세계의 객체는 무한에 가까움

예를 들어, 학습 데이터에 “판다”가 없었다면 모델은 판다를 절대 탐지할 수 없습니다. 이 한계를 해결하기 위해 등장한 개념이 Zero-Shot Detection입니다.

###**2. Zero-Shot의 핵심 아이디어**
이미지와 텍스트를 같은 의미 공간(embedding space)에 매핑합니다. 이 접근은 멀티모달 사전학습 모델의 발전 덕분에 가능해졌습니다. 이 모델들은 이미지를 벡터로, 텍스트를 벡터로 변환하고, 의미가 유사하면 벡터 공간에서도 가깝게 위치하도록 학습됩니다.

###**3. Zero-Shot Object Detection 동작 구조**
~~~
이미지 → 박스 후보 생성
텍스트 → 텍스트 임베딩 생성
이미지 영역 임베딩 ↔ 텍스트 임베딩 유사도 계산
~~~

###**4. Open-Vocabulary**
Open-Vocabulary란 모델이 학습할 때 정해진 고정 클래스 집합에만 의존하지 않고, 텍스트로 새롭게 정의되는 개념까지 유연하게 인식·예측할 수 있는 능력을 의미합니다. 기존 모델은 학습 데이터에 포함된 클래스만 분류하거나 탐지할 수 있지만, Open-Vocabulary 모델은 이미지와 텍스트를 같은 의미 공간에 매핑하는 멀티모달 학습을 기반으로, 학습 시 보지 못한 객체나 개념도 텍스트 설명만으로 인식할 수 있습니다. 즉, 사전에 닫힌(class-closed) 구조가 아니라, 현실 세계처럼 계속 확장 가능한 열린(open) 개념 체계를 다루는 접근 방식입니다.

#**2. DETR**
DETR(DEtection TRansformer)은 Transformer 기반의 End-to-End 객체 탐지 모델로, 이미지에서 추출한 특징을 Transformer Encoder–Decoder 구조에 입력하고 학습 가능한 Object Query를 통해 고정 개수의 객체를 직접 예측합니다. 각 Query는 하나의 객체에 대응하며, 클래스 확률과 바운딩 박스를 동시에 출력합니다. 예측 결과는 집합(set) 단위로 처리되며, Hungarian Matching을 이용해 정답 객체와 1:1 매칭하여 학습합니다. 전역 Self-Attention을 활용해 이미지 전체 문맥 정보를 반영할 수 있으며, 탐지 과정을 하나의 통합된 네트워크로 구성한 것이 핵심 특징입니다. [논문](https://arxiv.org/pdf/2005.12872)

<img src='https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdna%2FE404X%2FdJMb99ZM1fs%2FAAAAAAAAAAAAAAAAAAAAAMtF6RXqJj8_77FycuC5V_lcwiNsEqY1qJocHG6NaMzF%2Fimg.png%3Fcredential%3DyqXZFxpELC7KVnFOS48ylbz2pIh7yKj8%26expires%3D1774969199%26allow_ip%3D%26allow_referer%3D%26signature%3Dc5ejRB5RUmr2TKLI%252FUkv6h%252FB7WI%253D'>

###**1. Object Query**
Object Query는 DETR에서 사용하는 학습 가능한 질의 벡터로, 이미지 안의 객체를 하나씩 예측하기 위한 “슬롯(slot)” 역할을 합니다. 미리 정해진 개수(예: 100개)의 Query가 Transformer Decoder에 입력되어, 각각이 하나의 객체 또는 “no object”를 담당하도록 학습됩니다. 각 Query는 이미지 특징과 상호작용하면서 클래스 확률과 바운딩 박스를 동시에 출력하며, 고정된 앵커 박스 대신 객체를 찾는 역할을 수행합니다. 즉, Object Query는 객체 후보를 생성하는 기준점이 아니라, 객체를 직접 예측하도록 학습되는 추상적인 질문 벡터라고 이해할 수 있습니다.

###**2. Hungarian Matching**
Hungarian Matching은 DETR에서 모델이 예측한 객체 집합과 실제 정답 객체 집합을 1:1로 최적으로 매칭하기 위해 사용하는 알고리즘입니다. DETR은 고정 개수의 객체를 한 번에 예측하기 때문에, 어떤 예측이 어떤 정답에 해당하는지를 정해줘야 학습이 가능합니다. 이때 각 예측과 정답 사이의 비용(예: 분류 손실, 바운딩 박스 L1 오차, IoU 손실 등)을 계산하고, 전체 비용의 합이 최소가 되도록 예측과 정답을 한 번씩만 연결해주는 방식이 Hungarian Matching입니다. 이를 통해 중복 예측을 제거하고, 별도의 NMS 없이도 집합 단위로 안정적인 학습이 가능해집니다.

#**3. DINO**
DINO(Detection with Improved DeNoising Anchor Boxes)는 DETR 계열의 객체 탐지 모델을 개선한 Transformer 기반 탐지 기법으로, 느린 수렴 문제와 학습 불안정을 해결하기 위해 denoising 학습 전략과 개선된 쿼리 초기화 방식을 도입한 모델입니다. 기존 DETR이 Hungarian Matching 기반의 집합 예측 구조를 유지하면서도, 정답 박스를 일부 노이즈를 섞은 형태로 함께 학습시켜 모델이 빠르게 정답 위치를 학습하도록 돕습니다. 또한 더 정교한 쿼리 설계와 학습 안정화 기법을 통해 작은 객체와 복잡한 장면에서도 높은 성능을 보이며, 현재 DETR 계열 모델 중 성능과 수렴 속도 면에서 매우 강력한 모델로 평가받고 있습니다.

###**1. DETR의 한계**
- Hungarian Matching 기반의 집합 예측은 초반 매칭이 불안정함
- Object Query가 어디를 봐야 하는지 모른 채 학습 시작
- 보통 500 epoch 이상 학습 필요
- 초기 위치 수렴이 느려 작은 객체에서 성능 저하
- 초반에 잘못된 매칭이 반복되면 학습이 흔들림
- loss가 안정적으로 감소하지 않는 구간 발생

###**2. DINO 구조**
DINO는 기본적으로 DETR 계열의 Transformer 기반 객체 탐지 모델입니다.
~~~
이미지
  ↓
Backbone (ResNet / Swin 등)
  ↓
멀티스케일 특징
  ↓
Transformer Encoder
  ↓
Transformer Decoder (Object Query + Denoising Query)
  ↓
클래스 + Bounding Box 예측
  ↓
Hungarian Matching + Denoising Loss
~~~

###**3. DINO Denoising Training 도입 및 강화**
정답 박스를 복사해서 좌표에 노이즈 추가해 일부 라벨 교란하고 이를 Query로 함께 입력합니다. 그리고 모델이 노이즈가 섞인 입력을 원래 정답으로 복원하도록 학습시킵니다.

👉 DETR의 가장 큰 문제였던 느린 학습을 해결

###**4. DINO Query/Reference(Anchor-like) 설계 개선**
- 더 구조화된 reference point 사용
- anchor-like box 표현 활용
- iterative refinement 구조 강화

👉 박스 위치가 더 빠르게 수렴하고 작은 객체에서 위치 정확도 향상 시켰으며, 복잡한 장면에서 안정적 탐지합니다.

###**💬 Swin Transformer**
Swin Transformer는 Vision Transformer를 계층적(hierarchical) 구조로 확장한 모델로, 이미지를 작은 패치 단위로 나눈 뒤 고정된 윈도우(window) 내부에서 self-attention을 수행하고, 층이 깊어질수록 윈도우를 이동(shift)시켜 전역 정보를 점진적으로 통합하는 방식의 비전 백본입니다. 기존 ViT가 전체 패치에 대해 전역 attention을 계산해 연산량이 큰 반면, Swin은 지역 윈도우 기반 attention을 사용해 계산 효율을 크게 줄이면서도 계층적으로 특징 맵 해상도를 줄여 CNN과 유사한 피라미드 구조를 형성합니다. 이 덕분에 분류뿐 아니라 객체 탐지·세그멘테이션 같은 멀티스케일 정보를 필요로 하는 작업에서 강력한 성능을 보이는 Transformer 기반 비전 모델입니다. [논문](https://arxiv.org/pdf/2103.14030)

<img src = 'https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdna%2FB91fs%2FdJMcafTdAJA%2FAAAAAAAAAAAAAAAAAAAAAFYTgZDfrMIrc5aTlOs5s2A9oqetXmSk_IMw5HsV5IDJ%2Fimg.png%3Fcredential%3DyqXZFxpELC7KVnFOS48ylbz2pIh7yKj8%26expires%3D1774969199%26allow_ip%3D%26allow_referer%3D%26signature%3DPX52%252FblOABJkTbumv9tsl3%252BFMbc%253D'>

#**4. GroundingDINO**
GroundingDINO는 DINO 기반의 Transformer 객체 탐지 구조에 텍스트 인코더를 결합해, 이미지와 텍스트를 동시에 이해하고 텍스트 조건에 맞는 객체를 탐지할 수 있도록 확장한 Open-Vocabulary 탐지 모델입니다. 즉, 고정된 클래스 집합에 의존하지 않고 사용자가 입력한 문장이나 단어(예: “a red car”, “a firefighter”)를 조건으로 받아, 해당 표현과 의미적으로 일치하는 객체의 위치를 바운딩 박스로 예측합니다. 내부적으로는 이미지 특징과 텍스트 임베딩을 공동 공간에서 정렬하고, cross-attention을 통해 텍스트 정보가 탐지 과정에 직접 반영되도록 설계되어 있어, 학습 시 보지 못한 객체도 텍스트 설명만으로 탐지할 수 있다는 점이 핵심 특징입니다. [논문](https://arxiv.org/pdf/2303.05499)

<img src='https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdna%2FK0qwN%2FdJMcafFGOV1%2FAAAAAAAAAAAAAAAAAAAAAOU_2iWxGXsfZISvUctzrEW7GeXxHzoD5_02mmYG2paZ%2Fimg.webp%3Fcredential%3DyqXZFxpELC7KVnFOS48ylbz2pIh7yKj8%26expires%3D1774969199%26allow_ip%3D%26allow_referer%3D%26signature%3DoJfLBhy3gIITnXgDvqp%252Fs9dwbKo%253D'>

###**1. 문장으로 객체를 찾기**
Grounding DINO는 전통적인 객체 탐지처럼 “사전에 정해진 라벨 목록(예: COCO 80개)”만 맞추는 게 아니라, 사용자가 준 텍스트(단어/문장/구문)에 해당하는 객체를 이미지에서 찾아 박스로 반환하는 모델입니다.



예를 들어 입력이 아래와 같이 들어오면

- 이미지: 고양이 + 리모컨이 있는 사진
- 텍스트: "a cat. a remote control."

출력은:

- "cat"에 해당하는 박스(들)
- "remote control"에 해당하는 박스(들)
- 각 박스의 점수(score)

즉, 텍스트 기반 조건부 탐지(Text-conditioned Detection) 또는 Open-Vocabulary / Zero-shot Detection을 수행합니다.

###**2. “Grounding”이라는 말의 의미**
여기서 “Grounding(접지/정착)”은 언어(단어/구문)를 이미지의 특정 영역(박스)에 연결하는 것
을 뜻합니다.
- “cat”이라는 단어가 이미지의 어디를 가리키는지
- “remote control”이라는 구문이 이미지의 어디를 가리키는지

이 연결을 학습으로 만들어냅니다. 그래서 Grounding DINO는 “분류(classification)”보다는
문구(phrase)와 영역(region)을 매칭하는 능력이 핵심입니다.

In [ ]:
import requests
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

model_id = "IDEA-Research/grounding-dino-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)
model.eval()

image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

text = "a cat. a remote control."

inputs = processor(images=image, text=text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# 후처리: 모델 출력 -> 박스/라벨/점수
results = processor.post_process_grounded_object_detection(
    outputs=outputs,
    input_ids=inputs.input_ids,
    threshold=0.28,
    text_threshold=0.28,
    target_sizes=[image.size[::-1]]  # (H, W)
)


fig, ax = plt.subplots(1, figsize=(10, 8))
ax.imshow(image)


for score, box, label in zip(
    results[0]["scores"].cpu(),
    results[0]["boxes"].cpu(),
    results[0]["text_labels"]
):
    x_min, y_min, x_max, y_max = box.tolist()
    width, height = x_max - x_min, y_max - y_min

    rect = patches.Rectangle(
        (x_min, y_min),
        width,
        height,
        linewidth=2,
        edgecolor="red",
        facecolor="none"
    )
    ax.add_patch(rect)

    caption = f"{label} ({score:.2f})"
    ax.text(
        x_min,
        max(0, y_min - 5),
        caption,
        fontsize=12,
        color="white",
        bbox=dict(facecolor="red", alpha=0.5, edgecolor="none", boxstyle="round,pad=0.3")
    )

ax.axis("off")

out_path = "test.png"
plt.tight_layout()
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved: {out_path}")